# Suite TS5CHK — Checking TypeScript 5 programs against versions, and as trees

Which versions of TypeScript 5 have which constructs, as the port records it: `TypeScript5.Standards` records the
versions `Programs.TypeScript5` covers as data, TypeScript 5.0 to 5.9, as the user decided ("Use Typescript 5.0-5.9 to
version the typescript5 standards."), each holding the statement, clause and member kinds it has, as mbse-expressions'
dialects record the terms' (mbse-basis CAP-268); `TypeScript5.Check.check(store, path, version)` lists the constructs
a stored program uses that a version lacks, each located by where it stands in the program's tree (`body[0]`,
`body[1].declarations[0].init`), reading those standards and, for what a property's value makes, its own `FEATURES`;
`TypeScript5.Check.validate(store, path)` lists a program's problems as a tree, each kind's own rules and its terms' as
their dialects validate them. Every kind is in every version of TypeScript 5, so what `check` finds is what a
property's value makes; JSX is an extension, outside every version. Where a statement, a clause or a member may stand
is no rule of `validate`'s: the schemas refuse a misplaced one on write. Ported from suite TS (each kind's own checks,
TS-03, and where features exist, TS-04, within TypeScript 5) and TSPRT (printing checked the standard).

A program's tree is one value, its `body`, each statement a value object; where a term holds statements, a
function's or an arrow's `block` and a class expression's `classBody`, `check` and `validate` read them as
statements, located through the term's place. Each case parses source with `TypeScript5.Parse.parse` into one store,
`TypeScript5.store()` prepared by `Standards.prepare`, or, where the parser would not write the tree, writes the
program's `body` by hand in mbse-schemas' builder syntax, as `Parse` writes it: a list of branches of
`Programs.TypeScript5.Statement`, each tagged by its kind in lower case (`ifstatement`), the terms nested in it as
values of mbse-expressions' unions (`a` below writes the identifier `a`).

In [ ]:
import itertools
from pathlib import Path

from mbse.Expressions.Framework import Standards as ExpressionStandards
from mbse.Expressions.TypeScript.TypeScript5 import Expressions as TypeScriptExpressions
from mbse.Programs import TypeScript5
from mbse.Programs.Framework import Model
from mbse.Programs.TypeScript5 import Check, Parse, Print, Standards
from mbse.Programs.TypeScript5.Statements import KINDS, LANGUAGE
from support import raises

SOURCES = Path("../../conformance/sources")
STORE = Standards.prepare(TypeScript5.store())
PATHS = (f"case{n}" for n in itertools.count())


def parsed(text, extension=".ts"):
    """The path of a new program holding `text`, parsed into the store; `extension` decides whether JSX is read."""
    path = next(PATHS) + extension
    STORE.transaction(Parse.parse(text, path))
    return path


def check(text, version):
    return Check.check(STORE, parsed(text), version)


def needs(text, minor, *problems):
    """`text` is fine in TypeScript 5.`minor`, and has `problems` in the version before."""
    path = parsed(text)
    assert Check.check(STORE, path, (5, minor)) == [], Check.check(STORE, path, (5, minor))
    assert Check.check(STORE, path, (5, minor - 1)) == list(problems), Check.check(STORE, path, (5, minor - 1))


def written(*body):
    """The path of a new program whose `body` is the statements `body`, each written in builder syntax as a branch of
    `Programs.TypeScript5.Statement`, as `Parse.parse` writes one."""
    path = next(PATHS) + ".ts"

    def transaction(txn):
        return txn.uid(path).mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
            .symbols({"ws": txn.Programs.Framework.Workspace, "program": txn.Programs.TypeScript5.Program})
            .spec([
                lambda _: _.ws.files({path: lambda e: e.file(_.program)}),
                lambda _: _.program.body(list(body)),
            ])]))])

    STORE.transaction(transaction)
    return path


def validate(*body):
    return Check.validate(STORE, written(*body))


# Terms and statements the cases share, in the builders' syntax: three names and a type, terms; `;` and `b;`
a = lambda t: t.identifier(lambda i: i.name("a"))  # noqa: E731
b = lambda t: t.identifier(lambda i: i.name("b"))  # noqa: E731
c = lambda t: t.identifier(lambda i: i.name("c"))  # noqa: E731
T = lambda t: t.tstypereference(lambda r: r.typeName(lambda n: n.identifier(lambda i: i.name("T"))))  # noqa: E731
EMPTY = lambda u: u.emptystatement(lambda s: s)  # noqa: E731
SAY_B = lambda u: u.expressionstatement(lambda s: s.expression(b))  # noqa: E731
LET_A = lambda u: u.variabledeclaration(lambda s: s.declarationKind("let").declarations([lambda d: d.id(a)]))  # noqa: E731
M = lambda l: l.value("m").text('"m"').quote('"')  # noqa: E731  # a string, where a literal stands: a source
STRING = lambda t: t.literal(M)  # noqa: E731  # the same, where a term stands
JSON = lambda k: k.key(lambda t: t.identifier(lambda i: i.name("type"))).value(  # noqa: E731  # `type: "json"`
    lambda l: l.value("json").text('"json"').quote('"'))

## TS5CHK-01 · The versions as data: `Programs.TypeScript5` covers TypeScript 5.0 to 5.9, each holding every statement, clause and member kind, beside mbse-expressions' dialects' own; run again, nothing changes

In [ ]:
with STORE.transaction():
    language = STORE.singleton(LANGUAGE)
    standards = ExpressionStandards.standards(language)
    assert list(standards) == [f"TypeScript 5.{minor}" for minor in range(10)]
    assert ExpressionStandards.find(language, "TypeScript 5.9") is standards["TypeScript 5.9"]
    assert ExpressionStandards.properties(standards["TypeScript 5.2"]) == {"bodies": ("com.microsoft",),
                                                                           "revision": ("version", "5.2")}
    assert ExpressionStandards.designation_of(standards["TypeScript 5.0"]) == "TypeScript 5.0"

    def kinds(minor):
        return {kind.name.rsplit(".", 1)[-1] for kind in ExpressionStandards.kinds(standards[f"TypeScript 5.{minor}"])}

    assert len(KINDS) == 43 and all(kinds(minor) == set(KINDS) for minor in range(10))  # every kind in every version
    assert "Error" not in kinds(9)  # invalid syntax, in none
    # The terms' versions are mbse-expressions' own, each dialect's by the same designations
    for dialect in Standards.DIALECTS[1:]:
        assert list(ExpressionStandards.standards(STORE.singleton(dialect))) == list(standards), dialect
    expressions = ExpressionStandards.standards(STORE.singleton("Expressions.TypeScript.TypeScript5"))
    tags = {kind.name for kind in ExpressionStandards.kinds(expressions["TypeScript 5.0"])}
    assert "Expressions.TypeScript.TypeScript5.ImportExpression" in tags
    assert "Expressions.TypeScript.TypeScript5.JSXElement" not in tags  # JSX, an extension, in none
# The language is one of the framework's dialects, which is what may own a standard
dialects = STORE.registered("Expressions.Framework.Dialect")
assert any(b.name == "programs_typescript5" and b.schema.name == "Programs.TypeScript5.Language"
           for b in dialects.branches)
# Run again, each writes nothing: the same versions, each with the same kinds
Standards.prepare(STORE)
with STORE.transaction():
    again = ExpressionStandards.standards(STORE.singleton(LANGUAGE))
    assert list(again) == list(standards) and len(ExpressionStandards.kinds(again["TypeScript 5.9"])) == 43
assert Standards.designation(9) == "TypeScript 5.9" and list(Standards.MINORS) == list(range(10))
assert Standards.SINCE == {}  # every kind older than TypeScript 5.0
# What `define` writes from `SINCE`, `check` reads: a kind recorded as arriving later is lacking before
Standards.SINCE["AccessorProperty"] = 4
try:
    later = Standards.prepare(TypeScript5.store())
finally:
    del Standards.SINCE["AccessorProperty"]
later.transaction(Parse.parse("class C {\n    accessor a = 1;\n}\n", "later.ts"))
assert Check.check(later, "later.ts", (5, 4)) == []
assert Check.check(later, "later.ts", (5, 3)) == ["body[0].body.body[0]: AccessorProperty needs TypeScript 5.4"]
print("ok")

## TS5CHK-02 · Where features exist within TypeScript 5: since which version, each located by where it stands in the program's tree, each once; what arrived before 5.0, every version has

In [ ]:
needs("using a = b;\n", 2, "body[0]: VariableDeclaration.declarationKind 'using' needs TypeScript 5.2")
needs("await using a = b, c = d;\n", 2, "body[0]: VariableDeclaration.declarationKind 'await using' needs TypeScript 5.2")
needs("for (using x of y) {}\n", 2, "body[0].left: VariableDeclaration.declarationKind 'using' needs TypeScript 5.2")
needs("for (await using a = b, c = d; ;) {}\n", 2,
      "body[0].init: VariableDeclaration.declarationKind 'await using' needs TypeScript 5.2")
needs('import data from "./d.json" with { type: "json" };\n', 3,
      "body[0]: ImportDeclaration.attributes needs TypeScript 5.3")
# Attributes on an export, which tree-sitter-typescript 0.23 does not read, written
path = written(lambda u: u.exportnameddeclaration(lambda s: s.exportKind("value").specifiers([lambda k: k
                   .exportKind("value").local(a).exported(a)]).source(M).attributes([JSON])),
               lambda u: u.exportalldeclaration(lambda s: s.exportKind("value").source(M).attributes([JSON])))
assert Check.check(STORE, path, (5, 3)) == []
assert Check.check(STORE, path, (5, 2)) == ["body[0]: ExportNamedDeclaration.attributes needs TypeScript 5.3",
                                            "body[1]: ExportAllDeclaration.attributes needs TypeScript 5.3"]
needs('const m = import("m", { with: { type: "json" } });\n', 3,
      "body[0].declarations[0].init: importexpression.options needs TypeScript 5.3")
needs('f(import("a", { with: {} }), import("b", { with: {} }));\n', 3,  # one term: once
      "body[0].expression: importexpression.options needs TypeScript 5.3")
needs('type M = import("m", { with: { "resolution-mode": "import" } });\n', 3,
      "body[0].typeAnnotation: tsimporttype.options needs TypeScript 5.3")
needs('import { "a-b" as x } from "m";\n', 6,
      "body[0].specifiers[0]: ImportSpecifier.imported as a string needs TypeScript 5.6")
needs('export { "a-b" as "c-d" } from "m";\nexport { x as "y" };\nexport * as "z" from "m";\n', 6,
      "body[0].specifiers[0]: ExportSpecifier.local as a string needs TypeScript 5.6",
      "body[0].specifiers[0]: ExportSpecifier.exported as a string needs TypeScript 5.6",
      "body[1].specifiers[0]: ExportSpecifier.exported as a string needs TypeScript 5.6",
      "body[2]: ExportAllDeclaration.exported as a string needs TypeScript 5.6")
needs('import defer * as m from "m";\n', 9, "body[0]: ImportDeclaration.phase 'defer' needs TypeScript 5.9")
# `import.defer(...)`, which tree-sitter-typescript does not read, written: its phase arrived in 5.9
path = written(lambda u: u.expressionstatement(lambda s: s.expression(lambda e: e.importexpression(lambda i: i
    .phase("defer").source(STRING)))))
assert Check.check(STORE, path, (5, 9)) == []
assert Check.check(STORE, path, (5, 8)) == ["body[0].expression: importexpression.phase 'defer' needs TypeScript 5.9"]
# What arrived before TypeScript 5.0, every version has: `accessor`, static blocks, decorators, `satisfies`, `const`
# type parameters, `export type *`, `?.` and `??`, variance annotations, `import type` and `let`
assert check("class C {\n    accessor a = 1;\n    static {}\n    @d m() {}\n}\n"
             "let s = a satisfies T, o = a?.b ?? c;\n"
             'function f<const T>(x: T) {}\nexport type * from "m";\ninterface I<in out T> {}\n'
             'import type { X } from "m";\n', (5, 0)) == []
print("ok")

## TS5CHK-03 · The version by `(5, minor)` or by name; versions the language does not cover, a store not prepared, and a path the workspace does not hold

In [ ]:
path = parsed("using a = b;\n")
assert Check.check(STORE, path, (5, 1)) == Check.check(STORE, path, "TypeScript 5.1") == [
    "body[0]: VariableDeclaration.declarationKind 'using' needs TypeScript 5.2"]
assert Check.check(STORE, path, (5, 2)) == Check.check(STORE, path, "TypeScript 5.2") == []
for version in ((5, 10), (4, 9), (6, 0), "TypeScript 4.9", "5.2", "ES2025"):  # no ECMAScript edition
    with raises(LookupError, match="Programs.TypeScript5 covers no version"):
        Check.check(STORE, path, version)
with raises(LookupError):
    Check.check(STORE, "nowhere.ts", (5, 9))
with raises(LookupError):
    Check.validate(STORE, "nowhere.ts")
# Within the caller's transaction, a program's visitor is read; the store and a path open their own
with STORE.transaction():
    program = STORE.singleton(Model.WORKSPACE).files({"path": path})
    assert Check.check_file(STORE, program, (5, 1)) == [
        "body[0]: VariableDeclaration.declarationKind 'using' needs TypeScript 5.2"]
    assert Check.validate_file(program) == []
    with raises(ValueError, match="transactions do not nest"):
        Check.check(STORE, path, (5, 1))
# A store that holds the installs alone has no versions; one with the language's alone has no terms' versions
bare = TypeScript5.store()
bare.transaction(Parse.parse("using a = b;\n", "a.ts"))
with raises(LookupError, match="run Standards.prepare"):
    Check.check(bare, "a.ts", (5, 9))
bare.transaction(Standards.install)
bare.transaction(Standards.define)
with raises(LookupError, match="Expressions.TypeScript.TypeScript5 covers no version"):
    Check.check(bare, "a.ts", (5, 9))
assert Check.check(Standards.prepare(bare), "a.ts", (5, 2)) == []
print("ok")

## TS5CHK-04 · The conformance sources: none of their constructs lacking in TypeScript 5.9; in older versions, each construct where it arrived; JSX in none, an extension

In [ ]:
text = (SOURCES / "typescript59.ts").read_text(encoding="utf-8")
path = parsed(text)
assert Check.check(STORE, path, (5, 9)) == Check.check(STORE, path, "TypeScript 5.9") == []
assert Check.check(STORE, path, (5, 8)) == ["body[107]: ImportDeclaration.phase 'defer' needs TypeScript 5.9"]
oldest = Check.check(STORE, path, (5, 0))
assert oldest == [
    "body[4]: ImportDeclaration.attributes needs TypeScript 5.3",
    "body[31].body.body[2].declarations[0].init: importexpression.options needs TypeScript 5.3",  # in a function
    "body[105]: VariableDeclaration.declarationKind 'using' needs TypeScript 5.2",
    "body[106]: VariableDeclaration.declarationKind 'await using' needs TypeScript 5.2",
    "body[107]: ImportDeclaration.phase 'defer' needs TypeScript 5.9"], oldest
assert Check.check(STORE, path, (5, 2)) == [p for p in oldest if not p.endswith("5.2")]
assert Check.validate(STORE, path) == []
# JSX, an extension outside every version, which a `.tsx` file reads
jsx = parsed((SOURCES / "typescript59-jsx.tsx").read_text(encoding="utf-8"), ".tsx")
assert all(Check.check(STORE, jsx, (5, minor)) == [] for minor in range(10))
assert Check.validate(STORE, jsx) == []
assert Check.check(STORE, parsed("const e = <div>{() => { using a = b; }}</div>;\n", ".tsx"), (5, 0)) == [
    "body[0].declarations[0].init.children[0].expression.block.body[0]: "
    "VariableDeclaration.declarationKind 'using' needs TypeScript 5.2"]  # but what JSX holds is checked
print(len(oldest), "constructs lacking in TypeScript 5.0")

## TS5CHK-05 · Each kind's own rules: what it needs, what its lists must hold, what goes together, the words its properties may be, and the dangling `else`; where a statement, a clause or a member stands, the schemas enforce on write

A block (`body`, `consequent`) is a `Programs.TypeScript5.Body`, whose union has no clause branch; a `switch`'s `cases`
take only a case or an error, a `try`'s `handler` only a `catch` clause or an error, a class's body only members; a
kind has only its own properties. So a misplaced statement, clause or member is never stored: writing one fails, an
`AttributeError` for a branch the union does not have, and `validate` has no rule for it.

In [ ]:
DECLARE_A = lambda d: d.id(a)  # noqa: E731
problems = [
    (lambda u: u.comment(lambda s: s), "body[0]: a Comment needs a text"),
    (lambda u: u.expressionstatement(lambda s: s), "body[0]: an ExpressionStatement needs an expression"),
    (lambda u: u.variabledeclaration(lambda s: s.declarations([DECLARE_A])),
     "body[0]: a VariableDeclaration needs a declarationKind"),
    (lambda u: u.variabledeclaration(lambda s: s.declarationKind("let")),
     "body[0]: a VariableDeclaration needs a declarator"),
    (lambda u: u.variabledeclaration(lambda s: s.declarationKind("val").declarations([DECLARE_A])),
     "body[0]: VariableDeclaration.declarationKind cannot be 'val'"),
    (lambda u: u.variabledeclaration(lambda s: s.declarationKind("let").declarations([lambda d: d.init(a)])),
     "body[0].declarations[0]: a VariableDeclarator needs an id"),
    (lambda u: u.variabledeclaration(lambda s: s.declarationKind("let").declarations([lambda d: d.id(a).definite(True)])),
     "body[0].declarations[0]: definite needs a name with a type annotation"),
    (lambda u: u.ifstatement(lambda s: s.consequent(EMPTY)), "body[0]: an IfStatement needs a test"),
    (lambda u: u.ifstatement(lambda s: s.test(a)), "body[0]: an IfStatement needs a consequent"),
    (lambda u: u.whilestatement(lambda s: s.test(a)), "body[0]: a WhileStatement needs a body"),
    (lambda u: u.dowhilestatement(lambda s: s.body(EMPTY)), "body[0]: a DoWhileStatement needs a test"),
    (lambda u: u.forstatement(lambda s: s.test(a)), "body[0]: a ForStatement needs a body"),
    (lambda u: u.forinstatement(lambda s: s.right(a).body(EMPTY)), "body[0]: a ForInStatement needs a left"),
    (lambda u: u.forofstatement(lambda s: s.left(lambda l: l.pattern(a)).body(EMPTY)),
     "body[0]: a ForOfStatement needs a right"),
    (lambda u: u.withstatement(lambda s: s.body(EMPTY)), "body[0]: a WithStatement needs an object"),
    (lambda u: u.labeledstatement(lambda s: s.body(EMPTY)), "body[0]: a LabeledStatement needs a label"),
    (lambda u: u.switchstatement(lambda s: s), "body[0]: a SwitchStatement needs a discriminant"),
    (lambda u: u.throwstatement(lambda s: s), "body[0]: a ThrowStatement needs an argument"),
    (lambda u: u.trystatement(lambda s: s.block(lambda k: k)), "body[0]: a TryStatement needs a handler or a finalizer"),
    (lambda u: u.trystatement(lambda s: s.finalizer(lambda k: k)), "body[0]: a TryStatement needs a block"),
    (lambda u: u.trystatement(lambda s: s.block(lambda k: k).handler(lambda h: h.catchclause(lambda w: w))),
     "body[0].handler: a CatchClause needs a body"),
    (lambda u: u.functiondeclaration(lambda f: f.id(lambda i: i.name("f"))), "body[0]: a FunctionDeclaration needs a body"),
    (lambda u: u.classdeclaration(lambda k: k.id(lambda i: i.name("C"))), "body[0]: a ClassDeclaration needs a body"),
    (lambda u: u.classdeclaration(lambda k: k.body(lambda y: y.body([lambda m: m.methoddefinition(lambda d: d
        .methodKind("method").key(a))]))), "body[0].body.body[0]: a MethodDefinition needs a value"),
    (lambda u: u.classdeclaration(lambda k: k.body(lambda y: y.body([lambda m: m.methoddefinition(lambda d: d
        .methodKind("function").key(a).value(lambda f: f.tsemptybodyfunctionexpression(lambda f: f)))]))),
     "body[0].body.body[0]: MethodDefinition.methodKind cannot be 'function'"),
    (lambda u: u.classdeclaration(lambda k: k.body(lambda y: y.body([lambda m: m.propertydefinition(lambda p: p
        .static(True))]))), "body[0].body.body[0]: a PropertyDefinition needs a key"),
    (lambda u: u.classdeclaration(lambda k: k.body(lambda y: y.body([lambda m: m.accessorproperty(lambda p: p
        .accessibility("internal").key(a))]))),
     "body[0].body.body[0]: AccessorProperty.accessibility cannot be 'internal'"),
    (lambda u: u.importdeclaration(lambda s: s.importKind("value")), "body[0]: an ImportDeclaration needs a source"),
    (lambda u: u.importdeclaration(lambda s: s.importKind("value").phase("source").source(M)),
     "body[0]: ImportDeclaration.phase cannot be 'source'"),
    (lambda u: u.importdeclaration(lambda s: s.importKind("value").source(M).specifiers([lambda k: k
        .importnamespacespecifier(lambda n: n)])), "body[0].specifiers[0]: an ImportNamespaceSpecifier needs a local"),
    (lambda u: u.importdeclaration(lambda s: s.importKind("value").source(M).attributes([lambda k: k.key(a)])),
     "body[0].attributes[0]: an ImportAttribute needs a value"),
    (lambda u: u.exportnameddeclaration(lambda s: s.exportKind("value").specifiers([lambda k: k.exportKind("value")
        .local(a)])), "body[0].specifiers[0]: an ExportSpecifier needs an exported"),
    (lambda u: u.exportdefaultdeclaration(lambda s: s), "body[0]: an ExportDefaultDeclaration needs a declaration"),
    (lambda u: u.exportalldeclaration(lambda s: s.exportKind("both").source(M)),
     "body[0]: ExportAllDeclaration.exportKind cannot be 'both'"),
    (lambda u: u.tsimportequalsdeclaration(lambda s: s.importKind("value").id(lambda i: i.name("x"))),
     "body[0]: a TSImportEqualsDeclaration needs a moduleReference"),
    (lambda u: u.tstypealiasdeclaration(lambda s: s.id(lambda i: i.name("X"))),
     "body[0]: a TSTypeAliasDeclaration needs a typeAnnotation"),
    (lambda u: u.tsinterfacedeclaration(lambda s: s.id(lambda i: i.name("I")).body(lambda y: y).extends([lambda h: h])),
     "body[0].extends[0]: a TSInterfaceHeritage needs an expression"),
    (lambda u: u.tsenumdeclaration(lambda s: s.id(lambda i: i.name("E")).body(lambda y: y.members([
        lambda m: m.tsenummember(lambda e: e.initializer(a))]))), "body[0].body.members[0]: a TSEnumMember needs an id"),
    (lambda u: u.tsmoduledeclaration(lambda s: s.moduleKind("package").id(lambda n: n.identifier(lambda i: i.name("N")))),
     "body[0]: TSModuleDeclaration.moduleKind cannot be 'package'"),
]
for body, problem in problems:
    assert validate(body) == [problem], (problem, validate(body))
# `if (a) if (b) ; else ;` would give the else to the inner `if`, through loops, labels and `with` too
dangling = "body[0]: an if without else ends the consequent, and would take the else"
for consequent in (lambda k: k.ifstatement(lambda s: s.test(b).consequent(EMPTY)),
                   lambda k: k.labeledstatement(lambda s: s.label(lambda i: i.name("l")).body(lambda k: k.forofstatement(
                       lambda f: f.left(lambda l: l.pattern(c)).right(c).body(
                           lambda k: k.ifstatement(lambda s: s.test(b).consequent(EMPTY)))))),
                   lambda k: k.ifstatement(lambda s: s.test(b).consequent(EMPTY).alternate(lambda k: k.withstatement(
                       lambda w: w.object(c).body(lambda k: k.ifstatement(lambda s: s.test(c).consequent(EMPTY))))))):
    assert validate(lambda u: u.ifstatement(lambda s: s.test(a).consequent(consequent).alternate(EMPTY))) == [dangling]
for consequent in (lambda k: k.whilestatement(lambda s: s.test(b).body(EMPTY)),
                   lambda k: k.ifstatement(lambda s: s.test(b).consequent(EMPTY).alternate(EMPTY))):
    assert validate(lambda u: u.ifstatement(lambda s: s.test(a).consequent(consequent).alternate(EMPTY))) == []
assert validate(lambda u: u.ifstatement(lambda s: s.test(a).consequent(lambda k: k.ifstatement(
    lambda s: s.test(b).consequent(EMPTY))))) == []  # no else to take
assert validate(lambda u: u.ifstatement(lambda s: s.test(a).consequent(lambda k: k.whilestatement(
    lambda s: s.test(b))).alternate(EMPTY))) == ["body[0].consequent: a WhileStatement needs a body"]
assert validate(lambda u: u.trystatement(lambda s: s.block(lambda k: k).finalizer(lambda k: k))) == []
assert validate(lambda u: u.variabledeclaration(lambda s: s.declarationKind("let").declarations([lambda d: d.id(
    lambda p: p.identifier(lambda i: i.name("x").typeAnnotation(lambda n: n.typeAnnotation(T)))).definite(True)]))) == []
# Where a statement, a clause or a member stands, the schemas enforce: a case, a member, a statement out of place
with raises(AttributeError, match="switchcase"):
    written(lambda u: u.switchcase(lambda w: w))
with raises(AttributeError, match="expressionstatement"):
    written(lambda u: u.classdeclaration(lambda k: k.body(lambda y: y.body([SAY_B]))))
with raises(AttributeError, match="catchclause"):
    written(lambda u: u.blockstatement(lambda s: s.body([lambda k: k.catchclause(lambda w: w)])))
print(len(problems), "problems")

## TS5CHK-06 · Terms as their dialects validate them, located at the outermost term, helper or part that holds them; a helper or a kind's record a statement holds alone, by its own checks; names they refer to taken as bound

In [ ]:
one = lambda t: t.tsuniontype(lambda u: u.types([T]))  # noqa: E731
assert validate(lambda u: u.expressionstatement(lambda s: s.expression(lambda e: e.sequenceexpression(
    lambda q: q.expressions([a]))))) == ["body[0].expression: a sequenceexpression needs two expressions"]
assert validate(lambda u: u.returnstatement(lambda s: s.argument(lambda e: e.templateliteral(lambda q: q.quasis([
    lambda e: e.raw("a")]).expressions([b]))))) == [
    "body[0].argument: a templateliteral needs one more quasi than expressions"]
assert validate(lambda u: u.tstypealiasdeclaration(lambda s: s.id(lambda i: i.name("X")).typeAnnotation(one))) == [
    "body[0].typeAnnotation: a tsuniontype needs two types"]
assert validate(lambda u: u.variabledeclaration(lambda s: s.declarationKind("let").declarations([
    lambda d: d.id(a).init(lambda e: e.sequenceexpression(lambda q: q.expressions([b])))]))) == [
    "body[0].declarations[0].init: a sequenceexpression needs two expressions"]  # within a part
# Helpers and kinds' records a statement holds alone: type parameters, an annotation, a decorator, a class's name, an
# interface's member, each by its own checks and its terms'
assert validate(lambda u: u.tstypealiasdeclaration(lambda s: s.id(lambda i: i.name("X")).typeParameters(lambda d: d)
                                                   .typeAnnotation(T))) == [
    "body[0].typeParameters: a tstypeparameterdeclaration needs a type parameter"]
assert validate(lambda u: u.functiondeclaration(lambda f: f.id(lambda i: i.name("f")).returnType(
    lambda n: n.typeAnnotation(one)).body(lambda k: k))) == ["body[0].returnType: a tsuniontype needs two types"]
assert validate(lambda u: u.classdeclaration(lambda k: k.decorators([lambda d: d.expression(
    lambda e: e.sequenceexpression(lambda q: q.expressions([a])))]).id(lambda i: i.name("1C")).body(lambda y: y))) == [
    "body[0].decorators[0]: a sequenceexpression needs two expressions",
    "body[0].id: an identifier's name cannot be '1C'"]
assert validate(lambda u: u.tsinterfacedeclaration(lambda s: s.id(lambda i: i.name("I")).body(lambda y: y.body([
    lambda m: m.tspropertysignature(lambda p: p.key(a).typeAnnotation(lambda n: n.typeAnnotation(one)))])))) == [
    "body[0].body.body[0]: a tsuniontype needs two types"]
# Parsed programs: names are the program's to bind, so a term's names are taken as bound
assert Check.validate(STORE, parsed("x + y;\nf(a, ...b);\nclass C<T> extends D<T> implements I {\n"
                                    "    m(this: C<T>, @p q = r): T { return q; }\n}\n")) == []
print("ok")

## TS5CHK-07 · Printing writes what the store holds; checking it against a version is `check`'s, and validating it `validate`'s (ported from TSPRT)

In [ ]:
path = parsed("await using a = b;\n")
assert Print.print_module(STORE, path) == "await using a = b;\n"  # printed, whatever the version
assert Check.check(STORE, path, (5, 1)) == [
    "body[0]: VariableDeclaration.declarationKind 'await using' needs TypeScript 5.2"]
path = parsed('import defer * as m from "m";\n')
assert Print.print_module(STORE, path) == 'import defer * as m from "m";\n'
assert Check.check(STORE, path, (5, 8)) == ["body[0]: ImportDeclaration.phase 'defer' needs TypeScript 5.9"]
path = written(lambda u: u.trystatement(lambda s: s.block(lambda k: k.body([SAY_B]))))
assert Print.print_module(STORE, path) == "try {\n    b;\n}\n"  # printed, though no try is without both
assert Check.validate(STORE, path) == ["body[0]: a TryStatement needs a handler or a finalizer"]
print("ok")

## TS5CHK-08 · Invalid syntax, which a parse keeps: `validate` reports each `Error` and each `error` and `missing` term, located; `check` skips them and checks what was recovered within an `Error`; an `Error` stands where a statement, a clause or a member may, and holds statements, the schemas refusing a clause it does not have

In [ ]:
# Each `Error` and each `error` and `missing` term, located; nothing else of a parsed file of good and bad statements
path = parsed("import x = = 1;\n"
              "let a = 1;\n"
              "function f(a) {\n"
              "    let y = (a + );\n"
              "}\n"
              "class C {\n"
              "    x = = 1;\n"
              "    m() {}\n"
              "}\n"
              "enum E { A = , B }\n"
              "switch (a) { case 1: x = = 1; }\n")
assert Check.validate(STORE, path) == [
    "body[0]: invalid syntax 'import x = ='",  # its `1;` read as a statement of its own, `body[1]`
    "body[3].body.body[0].declarations[0].init: expression: right: missing identifier",
    "body[4].body.body[0]: invalid syntax 'x = = 1'",
    "body[5].body.members[1]: invalid syntax '='",  # an enum member that does not parse
    "body[6].cases[0].consequent[0].expression: invalid syntax 'x = = 1'",
], Check.validate(STORE, path)
# A `using` declarator without its value, or a `using` where only an expression stands
assert Check.validate(STORE, parsed("using a = b, c;\nf(using d = e);\n")) == [
    "body[0]: invalid syntax 'using a = b, c;'", "body[1].expression: argument 0: invalid syntax 'using d = e'"]
# `check` skips invalid syntax, which no standard lists, and checks what was recovered within it
assert Check.check(STORE, path, (5, 0)) == []
recovered = parsed("if (a {\n    using q = r;\n}\n")
assert Check.validate(STORE, recovered) == ["body[0]: invalid syntax 'if (a {\\n    using q = r;\\n}'"]
assert Check.check(STORE, recovered, (5, 1)) == ["body[0].body[0]: VariableDeclaration.declarationKind 'using' needs "
                                                 "TypeScript 5.2"]
# An `Error` stands where a statement, a case, a catch clause, a member or an enum member may, and holds statements in
# its body, each checked and validated as one
USING = lambda u: u.variabledeclaration(lambda s: s.declarationKind("using").declarations([DECLARE_A]))  # noqa: E731
path = written(lambda u: u.switchstatement(lambda s: s.discriminant(a).cases([lambda k: k.error(lambda e: e.text("case"))])),
               lambda u: u.trystatement(lambda s: s.block(lambda k: k).handler(lambda h: h.error(lambda e: e
                   .text("catch (").body([USING])))),
               lambda u: u.classdeclaration(lambda k: k.body(lambda y: y.body([lambda m: m.error(lambda e: e.text("= ;"))]))),
               lambda u: u.tsenumdeclaration(lambda s: s.id(lambda i: i.name("E")).body(lambda y: y.members([
                   lambda m: m.error(lambda e: e.text("%"))]))),
               lambda u: u.error(lambda e: e.body([lambda u: u.trystatement(lambda s: s.block(lambda k: k))])))
assert Check.validate(STORE, path) == [
    "body[0].cases[0]: invalid syntax 'case'", "body[1].handler: invalid syntax 'catch ('",
    "body[2].body.body[0]: invalid syntax '= ;'", "body[3].body.members[0]: invalid syntax '%'",
    "body[4]: invalid syntax", "body[4].body[0]: a TryStatement needs a handler or a finalizer"]
assert Check.check(STORE, path, (5, 1)) == [
    "body[1].handler.body[0]: VariableDeclaration.declarationKind 'using' needs TypeScript 5.2"]
with raises(AttributeError, match="switchcase"):
    written(lambda u: u.error(lambda e: e.text("").body([lambda k: k.switchcase(lambda w: w)])))
print("ok")

## TS5CHK-09 · Statements within terms: a function expression's and an arrow's `block`, an object's or a class's method's, and a class expression's `classBody`, checked and validated as statements, located through the term's place, to any depth; mbse-expressions' dialect does not read them

In [ ]:
path = parsed("const f = function () { using a = b; };\n"
              "g(() => { await using c = d; });\n"
              "const o = { m() { using e = f; } };\n"
              "let K = class {\n"
              "    static { using q = r; }\n"
              "    m() {\n"
              "        return () => { const h = function () { using s = t; }; };\n"
              "    }\n"
              "};\n"
              "class C { m() { return import('m', { with: {} }); } }\n")
assert Check.check(STORE, path, (5, 1)) == [
    "body[0].declarations[0].init.block.body[0]: VariableDeclaration.declarationKind 'using' needs TypeScript 5.2",
    "body[1].expression.arguments[0].block.body[0]: VariableDeclaration.declarationKind 'await using' needs TypeScript 5.2",
    "body[2].declarations[0].init.properties[0].value.block.body[0]: VariableDeclaration.declarationKind 'using' needs "
    "TypeScript 5.2",
    "body[3].declarations[0].init.classBody.body[0].body[0]: VariableDeclaration.declarationKind 'using' needs "
    "TypeScript 5.2",
    "body[3].declarations[0].init.classBody.body[1].value.block.body[0].argument.block.body[0].declarations[0].init"
    ".block.body[0]: VariableDeclaration.declarationKind 'using' needs TypeScript 5.2",
    "body[4].body.body[0].value.block.body[0].argument: importexpression.options needs TypeScript 5.3"]
assert Check.check(STORE, path, (5, 3)) == [] and Check.validate(STORE, path) == []
# Validated as statements: a kind's rules, an `Error` and a term's problems within a block or a class body
path = written(
    lambda u: u.variabledeclaration(lambda s: s.declarationKind("const").declarations([lambda d: d.id(a).init(
        lambda e: e.functionexpression(lambda f: f.block(lambda k: k.blockstatement(lambda k: k.body([
            lambda u: u.trystatement(lambda s: s.block(lambda k: k)),
            lambda u: u.error(lambda e: e.text("x = = 1"))])))))])),
    lambda u: u.expressionstatement(lambda s: s.expression(lambda e: e.classexpression(lambda k: k.classBody(
        lambda y: y.classbody(lambda y: y.body([
            lambda m: m.methoddefinition(lambda d: d.methodKind("method").key(a)),
            lambda m: m.staticblock(lambda s: s.body([lambda u: u.expressionstatement(lambda s: s.expression(
                lambda e: e.sequenceexpression(lambda q: q.expressions([b]))))]))])))))),
    lambda u: u.expressionstatement(lambda s: s.expression(lambda e: e.callexpression(lambda x: x.callee(a).arguments([
        lambda t: t.arrowfunctionexpression(lambda f: f.block(lambda k: k.blockstatement(lambda k: k.body([
            lambda u: u.ifstatement(lambda s: s.consequent(EMPTY))]))))])))))
assert Check.validate(STORE, path) == [
    "body[0].declarations[0].init.block.body[0]: a TryStatement needs a handler or a finalizer",
    "body[0].declarations[0].init.block.body[1]: invalid syntax 'x = = 1'",
    "body[1].expression.classBody.body[0]: a MethodDefinition needs a value",
    "body[1].expression.classBody.body[1].body[0].expression: a sequenceexpression needs two expressions",
    "body[2].expression.arguments[0].block.body[0]: an IfStatement needs a test"]
# A term's dialect stops at its block (`Terms.boundaries`): what it holds is this module's to read
with STORE.transaction():
    program = STORE.singleton(Model.WORKSPACE).files({"path": path})
    from mbse.Expressions.Framework import Objects, Terms
    (_, declaration), = Objects.properties(Objects.properties(program)["body"].values[0]).items()
    init = Objects.properties(Objects.properties(declaration)["declarations"].values[0])["init"]
    assert Terms.kind_of(init)[0] == "functionexpression" and TypeScriptExpressions.DIALECT.validate(init) == []
print("ok")